# PyTorch Recommender Demo

This notebook is a lightweight companion to the main recommender example. Its purpose is to show practical familiarity with PyTorch on a small embedding-based recommender, not to build a production recommender.

## Setup

- Uses the same MovieLens 100k files as the main recommender notebook.
- Dependencies are managed through the repository `requirements.txt`.
- The model is a simple matrix-factorization style network with user and item embeddings.

## What This Demo Shows

- A compact PyTorch workflow for a simple embedding-based recommender.
- Explicit device placement so the notebook uses GPU if available and CPU otherwise.
- A clear manual training loop with validation monitoring and early stopping.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
pd.set_option("display.max_colwidth", 60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

Running on: cuda


## Data Loading

The notebook reuses `../data/u.data` and `../data/u.item`. If the files are missing, it downloads the official MovieLens 100k archive.

In [2]:
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
data_dir = Path("../data")
data_dir.mkdir(parents=True, exist_ok=True)

required_files = ["u.data", "u.item"]
if not all((data_dir / file_name).exists() for file_name in required_files):
    zip_path = data_dir / "ml-100k.zip"
    extract_dir = data_dir / "ml-100k"
    urlretrieve(MOVIELENS_URL, zip_path)
    with ZipFile(zip_path, "r") as zip_file:
        zip_file.extractall(data_dir)
    for file_name in required_files:
        source = extract_dir / file_name
        destination = data_dir / file_name
        if source.exists() and not destination.exists():
            destination.write_bytes(source.read_bytes())

ratings = pd.read_csv(
    data_dir / "u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
)
movies = pd.read_csv(
    data_dir / "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["item_id", "title"],
)

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")
ratings = ratings.merge(movies, on="item_id", how="left")
ratings.head()

,user_id,item_id,rating,timestamp,title
0,196,242,3,1997-12-04 15:55:49,Kolya (1996)
1,186,302,3,1998-04-04 19:22:22,L.A. Confidential (1997)
2,22,377,1,1997-11-07 07:18:36,Heavyweights (1994)
3,244,51,2,1997-11-27 05:02:03,Legends of the Fall (1994)
4,166,346,1,1998-02-02 05:33:16,Jackie Brown (1997)


## Preprocessing And Split

For consistency with the main notebook, we use a simple time-aware split and hold out the latest rating for each eligible user.

In [3]:
user_ids = np.sort(ratings["user_id"].unique())
item_ids = np.sort(ratings["item_id"].unique())
user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
item_to_index = {item_id: idx for idx, item_id in enumerate(item_ids)}

ratings["user_idx"] = ratings["user_id"].map(user_to_index)
ratings["item_idx"] = ratings["item_id"].map(item_to_index)

def make_time_split(frame, min_user_interactions=5):
    frame = frame.sort_values(["user_id", "timestamp", "item_id"]).copy()
    user_counts = frame.groupby("user_id")["item_id"].transform("size")
    eligible = frame[user_counts >= min_user_interactions]
    test_indices = eligible.groupby("user_id", sort=False).tail(1).index
    test = frame.loc[test_indices].copy().reset_index(drop=True)
    train = frame.drop(index=test_indices).reset_index(drop=True)
    return train, test

train_ratings, test_ratings = make_time_split(ratings)

validation_fraction = 0.1
validation_mask = np.random.rand(len(train_ratings)) < validation_fraction
validation_ratings = train_ratings.loc[validation_mask].reset_index(drop=True)
train_ratings = train_ratings.loc[~validation_mask].reset_index(drop=True)

print(f"Train size: {len(train_ratings):,}")
print(f"Validation size: {len(validation_ratings):,}")
print(f"Test size: {len(test_ratings):,}")

Train size: 89,120
Validation size: 9,937
Test size: 943


## Model

This PyTorch model learns one embedding vector per user and per movie, then predicts ratings from their dot product plus a global average.

In [4]:
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)

    def forward(self, user_idx, item_idx):
        user_vec = self.user_embedding(user_idx)
        item_vec = self.item_embedding(item_idx)
        scores = (user_vec * item_vec).sum(dim=1)
        return scores

n_users = len(user_ids)
n_items = len(item_ids)
embedding_dim = 32
global_mean = float(train_ratings["rating"].mean())
model = MatrixFactorization(n_users, n_items, embedding_dim).to(device)
model

MatrixFactorization(
  (user_embedding): Embedding(943, 32)
  (item_embedding): Embedding(1682, 32)
)

## Training

A few epochs are enough here because the notebook is meant to illustrate framework usage, not optimize a deep recommender.

In [5]:
centered_train_ratings = train_ratings["rating"] - global_mean
centered_validation_ratings = validation_ratings["rating"] - global_mean

train_dataset = TensorDataset(
    torch.tensor(train_ratings["user_idx"].to_numpy(), dtype=torch.long),
    torch.tensor(train_ratings["item_idx"].to_numpy(), dtype=torch.long),
    torch.tensor(centered_train_ratings.to_numpy(), dtype=torch.float32),
)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

validation_dataset = TensorDataset(
    torch.tensor(validation_ratings["user_idx"].to_numpy(), dtype=torch.long),
    torch.tensor(validation_ratings["item_idx"].to_numpy(), dtype=torch.long),
    torch.tensor(centered_validation_ratings.to_numpy(), dtype=torch.float32),
)
validation_loader = DataLoader(validation_dataset, batch_size=1024, shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
history = []
best_val_loss = float("inf")
best_state = None
patience = 2
epochs_without_improvement = 0

for epoch in range(20):
    model.train()
    epoch_losses = []
    for user_batch, item_batch, rating_batch in train_loader:
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        rating_batch = rating_batch.to(device)
        optimizer.zero_grad()
        predictions = model(user_batch, item_batch)
        loss = loss_fn(predictions, rating_batch)
        loss.backward()
        optimizer.step()
        epoch_losses.append(float(loss.item()))

    model.eval()
    validation_losses = []
    with torch.no_grad():
        for user_batch, item_batch, rating_batch in validation_loader:
            user_batch = user_batch.to(device)
            item_batch = item_batch.to(device)
            rating_batch = rating_batch.to(device)
            predictions = model(user_batch, item_batch)
            validation_losses.append(float(loss_fn(predictions, rating_batch).item()))

    train_mse = np.mean(epoch_losses)
    val_mse = np.mean(validation_losses)
    history.append({"epoch": epoch + 1, "train_mse": train_mse, "val_mse": val_mse})

    if val_mse < best_val_loss:
        best_val_loss = val_mse
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            break

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)

pd.DataFrame(history)

,epoch,train_mse,val_mse
0,1,25.777611,18.165366
1,2,12.296241,11.637130
2,3,7.037176,8.365821
3,4,4.519707,6.516722
4,5,3.161151,5.394788
5,6,2.390853,4.655978
6,7,1.913639,4.145942
7,8,1.596268,3.781033
8,9,1.386290,3.510374
9,10,1.225388,3.304360


## Evaluation

We keep evaluation simple and use RMSE on the held-out ratings.

In [6]:
model.eval()
with torch.no_grad():
    test_user_tensor = torch.tensor(test_ratings["user_idx"].to_numpy(), dtype=torch.long, device=device)
    test_item_tensor = torch.tensor(test_ratings["item_idx"].to_numpy(), dtype=torch.long, device=device)
    test_predictions = model(test_user_tensor, test_item_tensor).cpu().numpy() + global_mean

rmse = float(np.sqrt(np.mean((test_predictions - test_ratings["rating"].to_numpy()) ** 2)))
print(f"Test RMSE: {rmse:.4f}")

Test RMSE: 2.4389


Interpretation: lower `RMSE` means the predicted ratings stay closer to the held-out true ratings. This is a simple signal that the model is learning useful user-item patterns, even though rating accuracy is not the same as full recommendation quality.

## Example Recommendations

Below we score all unseen movies for one user and return the top recommendations.

In [7]:
movie_lookup = movies.set_index("item_id")

def recommend_for_user(user_id, k=10):
    user_idx = user_to_index[user_id]
    item_tensor = torch.arange(n_items, dtype=torch.long, device=device)
    user_tensor = torch.full((n_items,), fill_value=user_idx, dtype=torch.long, device=device)

    model.eval()
    with torch.no_grad():
        scores = model(user_tensor, item_tensor).cpu().numpy() + global_mean

    seen_indices = train_ratings.loc[train_ratings["user_id"] == user_id, "item_idx"].to_numpy()
    scores[seen_indices] = -np.inf

    top_indices = np.argsort(scores)[::-1][:k]
    top_item_ids = [item_ids[idx] for idx in top_indices]
    return pd.DataFrame(
        {
            "item_id": top_item_ids,
            "title": movie_lookup.loc[top_item_ids, "title"].values,
            "predicted_score": scores[top_indices],
        }
    )

example_user_id = int(test_ratings.iloc[0]["user_id"])
recommend_for_user(example_user_id, k=10)

,item_id,title,predicted_score
0,1434,Shooting Fish (1997),8.573062
1,1455,"Outlaw, The (1943)",8.342705
2,1646,Men With Guns (1997),8.199213
3,1616,Desert Winds (1995),8.182185
4,1096,Commandments (1997),7.986638
5,782,Little Odessa (1994),7.643441
6,1404,Withnail and I (1987),7.511415
7,1336,Kazaam (1996),7.502473
8,639,"Tin Drum, The (Blechtrommel, Die) (1979)",7.356102
9,1534,Twin Town (1997),7.297767


## What This Demo Does Not Cover

- Richer recommender setups such as implicit-feedback training, multi-stage retrieval and ranking, or metadata-driven models.
- Broader evaluation beyond `RMSE`, such as ranking metrics, diversity, or online testing.
- Production concerns such as experiment tracking, monitoring, and deployment.